# R001 vs R002 refined comparison

This notebook compares:
- `data/annotation_rounds/r001_gold_exports/r001_refined_combined/r001_refined.csv`
- `data/annotation_rounds/r002_gold_exports/r002_refined_combined/r002_refined.csv`

It covers:
- column/schema differences
- row counts and missingness
- verdict and action distributions (Gold/Bad/etc.)
- correction behavior (`corrected_action`, `secondary_action`)
- overlap between rounds by `(video_uid, timestamp_sec)` key

In [18]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

base = Path("/shared/ssd_14T/home/leopolddas/HAR_Lab_Initiative_AI")
r1_path = base / "data/annotation_rounds/r001_gold_exports/r001_refined_combined/r001_refined.csv"
r2_path = base / "data/annotation_rounds/r002_gold_exports/r002_refined_combined/r002_refined.csv"

r1 = pd.read_csv(r1_path)
r2 = pd.read_csv(r2_path)

print(f"R001 shape: {r1.shape}")
print(f"R002 shape: {r2.shape}")
print("\nR001 path:", r1_path)
print("R002 path:", r2_path)

R001 shape: (17193, 9)
R002 shape: (10162, 19)

R001 path: /shared/ssd_14T/home/leopolddas/HAR_Lab_Initiative_AI/data/annotation_rounds/r001_gold_exports/r001_refined_combined/r001_refined.csv
R002 path: /shared/ssd_14T/home/leopolddas/HAR_Lab_Initiative_AI/data/annotation_rounds/r002_gold_exports/r002_refined_combined/r002_refined.csv


In [7]:
# 1) Schema / column comparison
r1_cols = set(r1.columns)
r2_cols = set(r2.columns)

only_r1 = sorted(r1_cols - r2_cols)
only_r2 = sorted(r2_cols - r1_cols)
shared = sorted(r1_cols & r2_cols)

print(f"Columns in R001: {len(r1_cols)}")
print(f"Columns in R002: {len(r2_cols)}")
print(f"Shared columns:   {len(shared)}")

print("\nOnly in R001:")
print(only_r1 if only_r1 else "None")

print("\nOnly in R002:")
print(only_r2 if only_r2 else "None")

schema_df = pd.DataFrame({
    "column": sorted(r1_cols | r2_cols),
})
schema_df["in_r001"] = schema_df["column"].isin(r1_cols)
schema_df["in_r002"] = schema_df["column"].isin(r2_cols)
schema_df["dtype_r001"] = schema_df["column"].map(lambda c: str(r1[c].dtype) if c in r1_cols else "-")
schema_df["dtype_r002"] = schema_df["column"].map(lambda c: str(r2[c].dtype) if c in r2_cols else "-")

schema_df

Columns in R001: 11
Columns in R002: 23
Shared columns:   9

Only in R001:
['status_main', 'status_secondary']

Only in R002:
['dataset_rows_covered_if_validated', 'global_rank', 'high_priority_reason', 'high_priority_review', 'historical_bad_count', 'historical_bad_rate', 'historical_checked_count', 'inner_id', 'normalized_narration_stage2', 'project', 'secondary_action', 'task_id', 'verdict', 'watchlist_hit']


,column,in_r001,in_r002,dtype_r001,dtype_r002
0,action,True,True,object,object
1,batch,True,True,int64,int64
2,corrected_action,True,True,object,object
3,dataset_rows_covered_if_validated,False,True,-,int64
4,global_rank,False,True,-,int64
5,high_priority_reason,False,True,-,object
6,high_priority_review,False,True,-,object
7,historical_bad_count,False,True,-,int64
8,historical_bad_rate,False,True,-,float64
9,historical_checked_count,False,True,-,int64


In [3]:
# 2) Missingness and quick quality snapshot

def missing_table(df, name):
    out = df.isna().mean().sort_values(ascending=False).rename("missing_ratio").to_frame()
    out["dataset"] = name
    out["missing_pct"] = (out["missing_ratio"] * 100).round(2)
    return out[["dataset", "missing_pct"]]

m1 = missing_table(r1, "R001")
m2 = missing_table(r2, "R002")

print("Top missing columns in R001")
display(m1.head(10))
print("Top missing columns in R002")
display(m2.head(10))

print("\nDistinct videos:")
print("R001:", r1["video_uid"].nunique() if "video_uid" in r1.columns else "N/A")
print("R002:", r2["video_uid"].nunique() if "video_uid" in r2.columns else "N/A")

Top missing columns in R001


,dataset,missing_pct
status_secondary,R001,97.76
corrected_action,R001,85.34
status_main,R001,9.21
batch,R001,0.00
action,R001,0.00
narration_text,R001,0.00
reasoning,R001,0.00
scenario,R001,0.00
round,R001,0.00
timestamp_sec,R001,0.00


Top missing columns in R002


,dataset,missing_pct
secondary_action,R002,96.21
inner_id,R002,91.60
task_id,R002,91.60
project,R002,91.60
corrected_action,R002,86.84
high_priority_reason,R002,81.84
verdict,R002,0.17
reasoning,R002,0.00
action,R002,0.00
scenario,R002,0.00



Distinct videos:
R001: 1346
R002: 1159


In [4]:
# 3) Verdict and action distributions

def pct_counts(df, col):
    s = df[col].fillna("<NA>").astype(str).str.strip()
    vc = s.value_counts(dropna=False)
    out = vc.to_frame("count")
    out["pct"] = (100 * out["count"] / len(df)).round(2)
    return out

if "verdict" in r1.columns and "verdict" in r2.columns:
    verdict_cmp = pd.concat(
        {
            "R001": pct_counts(r1, "verdict"),
            "R002": pct_counts(r2, "verdict"),
        },
        axis=1,
    )
    print("Verdict distribution (Gold/Bad/etc.)")
    display(verdict_cmp)

if "action" in r1.columns and "action" in r2.columns:
    action_cmp = pd.concat(
        {
            "R001": pct_counts(r1, "action"),
            "R002": pct_counts(r2, "action"),
        },
        axis=1,
    ).fillna(0)
    print("Action distribution")
    display(action_cmp.sort_values(("R002", "count"), ascending=False).head(20))

if "scenario" in r1.columns and "scenario" in r2.columns:
    scenario_cmp = pd.concat(
        {
            "R001": pct_counts(r1, "scenario"),
            "R002": pct_counts(r2, "scenario"),
        },
        axis=1,
    ).fillna(0)
    print("Scenario distribution")
    display(scenario_cmp.sort_values(("R002", "count"), ascending=False))

Action distribution


R001           R002       
                      count    pct   count    pct
action                                           
Object Transfer      6792.0  38.74  5736.0  48.19
Stationary           3807.0  21.71  2715.0  22.81
Task Operation          0.0   0.00  2392.0  20.10
Locomotion           1931.0  11.01   794.0   6.67
Search                932.0   5.32   266.0   2.23
Essential Operation  4072.0  23.22     0.0   0.00

Scenario distribution


R001         R002       
                   count    pct count    pct
scenario                                    
Cooking             5259  29.99  3753  31.53
Cleaning            2872  16.38  1863  15.65
Mechanical Repair   2641  15.06  1588  13.34
Playing Instrument  1870  10.66  1551  13.03
Carpentry           1586   9.05  1097   9.22
Walking Outdoors    1705   9.72   997   8.38
Desk Work           1006   5.74   579   4.86
Gardening            595   3.39   475   3.99

In [5]:
# 4) Correction-related analysis

def non_empty_ratio(df, col):
    if col not in df.columns:
        return np.nan
    s = df[col].fillna("").astype(str).str.strip()
    return (s != "").mean() * 100

corr_summary = pd.DataFrame(
    {
        "metric": [
            "corrected_action non-empty %",
            "secondary_action non-empty %",
            "reasoning non-empty %",
            "high_priority_review == yes %",
            "watchlist_hit == yes %",
        ],
        "R001": [
            non_empty_ratio(r1, "corrected_action"),
            non_empty_ratio(r1, "secondary_action"),
            non_empty_ratio(r1, "reasoning"),
            (r1["high_priority_review"].fillna("").astype(str).str.lower().eq("yes").mean() * 100) if "high_priority_review" in r1.columns else np.nan,
            (r1["watchlist_hit"].fillna("").astype(str).str.lower().eq("yes").mean() * 100) if "watchlist_hit" in r1.columns else np.nan,
        ],
        "R002": [
            non_empty_ratio(r2, "corrected_action"),
            non_empty_ratio(r2, "secondary_action"),
            non_empty_ratio(r2, "reasoning"),
            (r2["high_priority_review"].fillna("").astype(str).str.lower().eq("yes").mean() * 100) if "high_priority_review" in r2.columns else np.nan,
            (r2["watchlist_hit"].fillna("").astype(str).str.lower().eq("yes").mean() * 100) if "watchlist_hit" in r2.columns else np.nan,
        ],
    }
)

corr_summary[["R001", "R002"]] = corr_summary[["R001", "R002"]].round(2)
corr_summary

,metric,R001,R002
0,corrected_action non-empty %,14.66,13.16
1,secondary_action non-empty %,NaN,3.79
2,reasoning non-empty %,100.00,100.00
3,high_priority_review == yes %,NaN,18.16
4,watchlist_hit == yes %,NaN,17.97


In [6]:
# 5) Overlap between rounds by (video_uid, timestamp_sec)
# We round timestamp to milliseconds to reduce tiny float noise.

def make_key(df):
    if not {"video_uid", "timestamp_sec"}.issubset(df.columns):
        return pd.Series(dtype=str)
    uid = df["video_uid"].astype(str).str.strip()
    ts = pd.to_numeric(df["timestamp_sec"], errors="coerce").fillna(-1)
    ts_ms = (ts * 1000).round().astype("int64").astype(str)
    return uid + "__" + ts_ms

k1 = set(make_key(r1))
k2 = set(make_key(r2))
inter = k1 & k2
only1 = k1 - k2
only2 = k2 - k1

overlap_df = pd.DataFrame(
    {
        "metric": [
            "unique keys in R001",
            "unique keys in R002",
            "intersection keys",
            "R001-only keys",
            "R002-only keys",
            "overlap % of R001",
            "overlap % of R002",
        ],
        "value": [
            len(k1),
            len(k2),
            len(inter),
            len(only1),
            len(only2),
            round(100 * len(inter) / max(1, len(k1)), 2),
            round(100 * len(inter) / max(1, len(k2)), 2),
        ],
    }
)

overlap_df

,metric,value
0,unique keys in R001,17516.00
1,unique keys in R002,11882.00
2,intersection keys,22.00
3,R001-only keys,17494.00
4,R002-only keys,11860.00
5,overlap % of R001,0.13
6,overlap % of R002,0.19


In [7]:
# 6) Optional: quick side-by-side verdict x action cross-tab
if {"verdict", "action"}.issubset(r1.columns):
    print("R001 verdict x action")
    display(pd.crosstab(r1["verdict"], r1["action"], margins=True))

if {"verdict", "action"}.issubset(r2.columns):
    print("R002 verdict x action")
    display(pd.crosstab(r2["verdict"], r2["action"], margins=True))

R002 verdict x action


action,Locomotion,Object Transfer,Search,Stationary,Task Operation,All
verdict,,,,,,
Bad,239,287,72,780,202,1580
Delete Row,0,1,0,0,0,1
Gold,521,5415,184,1874,2168,10162
Skip,34,25,10,57,14,140
All,794,5728,266,2711,2384,11883


In [16]:
# Requested view: first 50 rows of R002 as a table
r2.head(50)
#global rank
#dataset_rows_covered_if_validated
#high_priority_review
#high_priority_reason	
#watchlist_hit
#historical_bad_count
#historical_checked_count	
#historical_bad_rate	
#task_id	
#project
#inner_id

,video_uid,timestamp_sec,narration_text,scenario,action,reasoning,batch,round,normalized_narration_stage2,dataset_rows_covered_if_validated,high_priority_review,high_priority_reason,watchlist_hit,historical_bad_count,historical_checked_count,historical_bad_rate,verdict,inner_id,secondary_action
0,05ac2c04-4a2c-4e8a-8b2a-2b60521b9d38,86.807300,#C C swings #unsure on the stick,Playing Instrument,Task Operation,Core task involving high hand activity.,1,r002,swings on stick,4,no,NaN,no,0,0,0.0,Gold,NaN,NaN
1,05ac2c04-4a2c-4e8a-8b2a-2b60521b9d38,178.718840,#C C moves thread ball,Playing Instrument,Task Operation,"Core task involving manual activity, likely re...",1,r002,moves thread ball,3,yes,watchlist_verb,yes,0,0,0.0,Gold,NaN,NaN
2,05ac2c04-4a2c-4e8a-8b2a-2b60521b9d38,313.663251,#C C crochets the textile,Playing Instrument,Task Operation,"Core manual task involving hands, high hand ac...",1,r002,crochets textile,5,no,NaN,no,0,0,0.0,Gold,NaN,NaN
3,05ac2c04-4a2c-4e8a-8b2a-2b60521b9d38,320.973011,#C C arranges the threads,Playing Instrument,Task Operation,Arranging threads is a fine motor task that li...,1,r002,arranges threads,4,no,NaN,no,0,0,0.0,Bad,NaN,NaN
4,06456897-960d-4d0c-8ce2-cd50a5a57bc3,331.922989,#C C picks the hedge shear,Gardening,Object Transfer,Logistics step of picking up an object.,1,r002,picks hedge shear,15,no,NaN,no,0,0,0.0,Gold,NaN,NaN
5,06456897-960d-4d0c-8ce2-cd50a5a57bc3,397.505199,#C C moves on the garden,Gardening,Locomotion,"Body moving through space, likely walking or m...",1,r002,moves on garden,5,yes,watchlist_verb,yes,0,0,0.0,Gold,NaN,NaN
6,06456897-960d-4d0c-8ce2-cd50a5a57bc3,464.450689,#C C plucks the hedge wall,Gardening,Task Operation,Body moving through space to maintain or trim ...,1,r002,plucks hedge wall,7,no,NaN,no,0,0,0.0,Bad,NaN,NaN
7,06456897-960d-4d0c-8ce2-cd50a5a57bc3,603.456638,#C C holds the rake,Gardening,Stationary,"Low body/hand motion, holding an object withou...",1,r002,holds rake,5,yes,watchlist_verb,yes,0,0,0.0,Gold,NaN,NaN
8,06456897-960d-4d0c-8ce2-cd50a5a57bc3,657.625288,#C C holds the pruning shear with both hands,Gardening,Stationary,"Low body/hand motion, holding an object.",1,r002,holds pruning shear with both hands,7,yes,watchlist_verb,yes,0,0,0.0,Gold,NaN,NaN
9,06456897-960d-4d0c-8ce2-cd50a5a57bc3,708.359048,#C C holds the cutter,Gardening,Stationary,The action describes holding an object without...,1,r002,holds cutter,6,yes,watchlist_verb,yes,0,0,0.0,Gold,NaN,NaN


In [19]:
# Requested view: first 100 rows of final HAR dataset
from pathlib import Path
import pandas as pd

har_path = Path('/shared/ssd_14T/home/leopolddas/HAR_Lab_Initiative_AI/data/annotation_rounds/final_gold_dataset/HAR_dataset.csv')
har_df = pd.read_csv(har_path)
har_df.head(100)

,video_uid,timestamp_sec,narration_text,scenario,action,reasoning,batch,round,status
0,000cd456-ff8d-499b-b0c1-4acead128a8b,311.614327,#C C moves hand towards the face,Cleaning,Stationary,"Low body/hand motion, likely a minor gesture o...",8,r001,Gold
1,000cd456-ff8d-499b-b0c1-4acead128a8b,452.731457,#C C picks a toy on the stool,Cleaning,Object Transfer,Logistics step involving picking up an object.,8,r001,Gold
2,001e3e4e-2743-47fc-8564-d5efd11f9e90,22.073549,#C C opens the washing machine door,Cleaning,Object Transfer,Logistics action involving opening a door.,6,r002,Gold
3,002ad105-bd9a-4858-953e-54e88dc7587e,146.444645,#C C rinses coriander,Cleaning,Essential Operation,Core manual task involving hand activity.,8,r001,Gold
4,002ad105-bd9a-4858-953e-54e88dc7587e,327.117935,#C C stands along countertop,Cleaning,Stationary,"Low body/hand motion, indicating a pause or id...",8,r001,Gold
...,...,...,...,...,...,...,...,...,...
95,002d2729-df71-438d-8396-5895b349e8fd,2455.684678,#C C pours sauce on the dough with the spoon i...,Cooking,Essential Operation,Core task involving high hand activity.,10,r001,Gold
96,002d2729-df71-438d-8396-5895b349e8fd,2485.737568,#C C opens the plate with her left hand,Cooking,Object Transfer,Logistics/Setup step involving picking up or p...,5,r002,Gold
97,002d2729-df71-438d-8396-5895b349e8fd,2486.455048,#C C drops the dough in the plate with her rig...,Cooking,Object Transfer,Refined via regex fallback,5,r002,Gold
98,002d2729-df71-438d-8396-5895b349e8fd,2488.777758,#C C adjusts the dough in the plate with her r...,Cooking,Task Operation,Core task involving high hand activity.,5,r002,Gold


In [11]:
r1.head(50)

,action,batch,narration_text,reasoning,round,scenario,status,timestamp_sec,video_uid
0,Locomotion,1,#c c walk around the streets,"Body moving through space, indicative of walking.",r001,Walking Outdoors,Gold,71.246610,02497a97-5de2-469c-b2e9-c940b78c2060
1,Locomotion,1,#C C negotiates the junction,"Body moving through space, navigating a juncti...",r001,Walking Outdoors,Gold,2196.219310,02497a97-5de2-469c-b2e9-c940b78c2060
2,Locomotion,1,#C C walks towards the ball,Body moving through space.,r001,Playing Instrument,Gold,46.474889,0296707b-0b42-45dd-98dc-ca8ce0a26a50
3,Search,1,#C C looks aside,"Head movement (visual search), hands mostly st...",r001,Playing Instrument,Gold,67.495689,0296707b-0b42-45dd-98dc-ca8ce0a26a50
4,Object Transfer,1,#C C picks shakers,Logistics/Setup step (picking up an object).,r001,Playing Instrument,Gold,80.469859,0550b3f4-94a5-4e91-a00f-316d4a1d4d59
5,Essential Operation,1,#C C plays #unsure,Core manual task involving high hand activity.,r001,Playing Instrument,Gold,489.041256,0550b3f4-94a5-4e91-a00f-316d4a1d4d59
6,Essential Operation,1,#C C shakes the maraca,Core task involving high hand activity.,r001,Playing Instrument,Gold,898.520408,0550b3f4-94a5-4e91-a00f-316d4a1d4d59
7,Object Transfer,1,#C C puts the maraca on the stand,"Logistics/Setup step, placing object on a surf...",r001,Playing Instrument,Gold,918.843798,0550b3f4-94a5-4e91-a00f-316d4a1d4d59
8,Essential Operation,1,#C C beats the drum,Core task involving repetitive hand movements.,r001,Playing Instrument,Gold,1846.943765,0550b3f4-94a5-4e91-a00f-316d4a1d4d59
9,Locomotion,1,#C C walks on the garden,"Movement through space, indicative of walking.",r001,Gardening,Gold,364.144239,06456897-960d-4d0c-8ce2-cd50a5a57bc3
